# Forecast Evolution and Nowcast Evaluation

**Module 03 — Notebook 03**  
**Estimated time:** 15 minutes

## Learning Objectives

1. Build a systematic forecast evolution chart tracking nowcast revisions quarter-by-quarter
2. Decompose the nowcast RMSE into contributions from each monthly vintage
3. Identify which quarters show the largest nowcast errors (COVID, 2008-09)
4. Evaluate whether the Beta MIDAS weight function is stable over time

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import beta as beta_dist
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Libraries loaded.")

## 1. Load Data

In [ ]:
USE_FRED = False

def load_series_from_csv(series_id):
    import os
    candidates = [
        '../resources',
        '../../module_00_foundations/resources',
        '../../../module_00_foundations/resources',
        '../../../../module_00_foundations/resources',
    ]
    filename_map = {
        'GDPC1':  'gdp_quarterly.csv',
        'INDPRO': 'industrial_production_monthly.csv',
    }
    fname = filename_map[series_id]
    for base in candidates:
        path = os.path.join(base, fname)
        if os.path.exists(path):
            df = pd.read_csv(path, index_col='date', parse_dates=True)
            return df.squeeze()
    raise FileNotFoundError(f"Cannot find {fname}")


if USE_FRED:
    from fredapi import Fred
    fred = Fred()
    gdp_raw = fred.get_series('GDPC1', observation_start='2000-01-01')
    ip_raw  = fred.get_series('INDPRO', observation_start='1999-01-01')
    gdp_growth = np.log(gdp_raw).diff().dropna() * 100
    ip_growth  = np.log(ip_raw).diff().dropna() * 100
else:
    gdp_growth = load_series_from_csv('GDPC1')
    ip_growth  = load_series_from_csv('INDPRO')

print(f"GDP: {len(gdp_growth)} quarters, IP: {len(ip_growth)} months")

In [ ]:
def build_midas_matrix_ragged(y_low_freq, x_high_freq, K, h_missing=0):
    if hasattr(y_low_freq.index, 'to_period'):
        y_q = y_low_freq.copy()
        y_q.index = y_low_freq.index.to_period('Q')
    else:
        y_q = y_low_freq.copy()
        y_q.index = pd.PeriodIndex(y_low_freq.index, freq='Q')

    if hasattr(x_high_freq.index, 'to_period'):
        x_m = x_high_freq.copy()
        x_m.index = x_high_freq.index.to_period('M')
    else:
        x_m = x_high_freq.copy()
        x_m.index = pd.PeriodIndex(x_high_freq.index, freq='M')

    K_avail = K - h_missing
    rows_Y, rows_X = [], []
    for q in y_q.index:
        last_avail = q.asfreq('M', how='end') - h_missing
        lags = [last_avail - i for i in range(K_avail)]
        if any(lag not in x_m.index for lag in lags): continue
        rows_Y.append(y_q[q])
        rows_X.append([x_m[lag] for lag in lags])
    return np.array(rows_Y), np.array(rows_X)


def beta_weights(K, theta1, theta2):
    if theta1 <= 0 or theta2 <= 0: return np.ones(K) / K
    x = (np.arange(K) + 0.5) / K
    raw = beta_dist.pdf(1 - x, theta1, theta2)
    s = raw.sum()
    return raw / s if s > 1e-12 else np.ones(K) / K


def estimate_midas(Y, X, starts=None):
    K = X.shape[1]
    if starts is None: starts = [(1.0, 5.0), (1.5, 4.0), (2.0, 3.0)]

    def psse(theta):
        t1, t2 = theta
        if t1 <= 0.01 or t2 <= 0.01: return 1e10
        w = beta_weights(K, t1, t2)
        xw = X @ w
        xc, yc = xw - xw.mean(), Y - Y.mean()
        ss = np.dot(xc, xc)
        if ss < 1e-12: return np.sum((Y - Y.mean())**2)
        b = np.dot(yc, xc) / ss
        a = Y.mean() - b * xw.mean()
        return np.sum((Y - a - b * xw)**2)

    best_sse, best_res = np.inf, None
    for t0 in starts:
        res = minimize(psse, t0, method='Nelder-Mead',
                       options={'maxiter': 20000, 'xatol': 1e-8})
        if res.fun < best_sse: best_sse, best_res = res.fun, res

    t1, t2 = max(best_res.x[0], 0.01), max(best_res.x[1], 0.01)
    w = beta_weights(K, t1, t2)
    xw = X @ w
    xc, yc = xw - xw.mean(), Y - Y.mean()
    beta_ = np.dot(yc, xc) / np.dot(xc, xc)
    alpha_ = Y.mean() - beta_ * xw.mean()
    resid = Y - alpha_ - beta_ * xw
    return {'theta1': t1, 'theta2': t2, 'alpha': alpha_, 'beta': beta_,
            'sse': best_sse, 'weights': w, 'residuals': resid}


K = 12
MIN_TRAIN = 30

Y_full, X_full = build_midas_matrix_ragged(gdp_growth, ip_growth, K, h_missing=0)
T = len(Y_full)
print(f"Data: T={T}, K={K}")

## 2. Full Expanding-Window Forecast Panel

Run expanding-window estimation for all three vintages and store the full error series (not just RMSE).

In [ ]:
# Build matrices for all three vintages
midas_matrices = {}
for h in [0, 1, 2]:
    Y_h, X_h = build_midas_matrix_ragged(gdp_growth, ip_growth, K, h_missing=h)
    midas_matrices[h] = (Y_h, X_h)

print("Running expanding-window forecast panel...")
print("(~2-3 minutes for all three vintages)")

# Store all forecasts and actuals
forecast_panel = {h: {'y_hat': [], 'y_act': [], 'error': []} for h in [0, 1, 2]}
# Also store estimated theta at each step for the complete-quarter vintage
theta_path = {'theta1': [], 'theta2': [], 'beta': []}

Y_base = midas_matrices[0][0]
n_oos = T - MIN_TRAIN

for t in range(MIN_TRAIN, T):
    y_actual = Y_base[t]

    for h in [0, 1, 2]:
        Y_h, X_h = midas_matrices[h]
        if t >= len(Y_h):
            forecast_panel[h]['y_hat'].append(np.nan)
            forecast_panel[h]['y_act'].append(y_actual)
            forecast_panel[h]['error'].append(np.nan)
            continue

        Y_tr = Y_h[:t]
        X_tr = X_h[:t]
        est = estimate_midas(Y_tr, X_tr, starts=[(1.0, 5.0), (1.5, 4.0)])
        w = beta_weights(X_tr.shape[1], est['theta1'], est['theta2'])
        xw_test = float(X_h[t] @ w)
        y_hat = est['alpha'] + est['beta'] * xw_test

        forecast_panel[h]['y_hat'].append(y_hat)
        forecast_panel[h]['y_act'].append(y_actual)
        forecast_panel[h]['error'].append(y_actual - y_hat)

        # Save theta path for h=0 (complete quarter)
        if h == 0:
            theta_path['theta1'].append(est['theta1'])
            theta_path['theta2'].append(est['theta2'])
            theta_path['beta'].append(est['beta'])

    if (t - MIN_TRAIN + 1) % 20 == 0:
        print(f"  Step {t - MIN_TRAIN + 1}/{n_oos}")

print("Done.")

# Compute RMSE summary
for h in [0, 1, 2]:
    errors = np.array(forecast_panel[h]['error'])
    valid = ~np.isnan(errors)
    rmse = np.sqrt(np.mean(errors[valid]**2))
    mae  = np.mean(np.abs(errors[valid]))
    vintage_name = {0: '3-month', 1: '2-month', 2: '1-month'}[h]
    print(f"  {vintage_name}: RMSE={rmse:.4f}, MAE={mae:.4f}")

## 3. Forecast Evolution — Quarter-by-Quarter

In [ ]:
# --- Build the full forecast evolution panel ---
fig, axes = plt.subplots(3, 1, figsize=(14, 11))

time_axis = np.arange(n_oos)
y_actual = np.array(forecast_panel[0]['y_act'])
y_hat_0  = np.array(forecast_panel[0]['y_hat'])
y_hat_1  = np.array(forecast_panel[1]['y_hat'])
y_hat_2  = np.array(forecast_panel[2]['y_hat'])

# Panel 1: Actual vs. nowcast series
ax = axes[0]
ax.plot(time_axis, y_actual, color='black', linewidth=1.5, zorder=5, label='Actual GDP growth')
ax.plot(time_axis, y_hat_0,  color='#1f77b4', linewidth=1.5, alpha=0.9,
        linestyle='-',  label='3-month nowcast')
ax.plot(time_axis, y_hat_1,  color='#ff7f0e', linewidth=1.2, alpha=0.8,
        linestyle='--', label='2-month nowcast')
ax.plot(time_axis, y_hat_2,  color='#2ca02c', linewidth=1.2, alpha=0.8,
        linestyle=':',  label='1-month nowcast')
ax.axhline(0, color='gray', linewidth=0.6, alpha=0.5)
ax.set_ylabel('GDP growth (%)')
ax.set_title('Actual GDP Growth vs. MIDAS Nowcasts')
ax.legend(fontsize=9, loc='upper right')

# Panel 2: Forecast errors
ax2 = axes[1]
err_0 = np.array(forecast_panel[0]['error'])
err_1 = np.array(forecast_panel[1]['error'])
err_2 = np.array(forecast_panel[2]['error'])

ax2.plot(time_axis, err_0, color='#1f77b4', linewidth=1.2, alpha=0.8, label='3-month error')
ax2.plot(time_axis, err_2, color='#2ca02c', linewidth=1.2, alpha=0.6,
         linestyle=':', label='1-month error')
ax2.axhline(0, color='black', linewidth=0.8)

# Shade error bands
ax2.fill_between(time_axis, err_0, 0, where=(np.array(err_0) > 0), alpha=0.15, color='blue')
ax2.fill_between(time_axis, err_0, 0, where=(np.array(err_0) < 0), alpha=0.15, color='red')
ax2.set_ylabel('Forecast error (%)')
ax2.set_title('Forecast Errors (Actual - Nowcast)')
ax2.legend(fontsize=9)

# Panel 3: Nowcast revisions (h=1→h=0 update)
ax3 = axes[2]
revision_12 = y_hat_1 - y_hat_2   # Adding month 2 to 1-month vintage
revision_01 = y_hat_0 - y_hat_1   # Adding month 3 to 2-month vintage

ax3.bar(time_axis - 0.2, revision_12, 0.38, color='#ff7f0e', alpha=0.8,
        label='Revision: adding month 2 (1-month → 2-month)')
ax3.bar(time_axis + 0.2, revision_01, 0.38, color='#1f77b4', alpha=0.8,
        label='Revision: adding month 3 (2-month → 3-month)')
ax3.axhline(0, color='black', linewidth=0.8)
ax3.set_xlabel('Out-of-sample observation index')
ax3.set_ylabel('Nowcast revision (%)')
ax3.set_title('Nowcast Revisions at Each Monthly IP Release')
ax3.legend(fontsize=9)

plt.suptitle(f'GDP Nowcast Forecast Evolution (K={K}, min_train={MIN_TRAIN})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('forecast_evolution.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved.")

## 4. Parameter Stability: Theta Path Over Time

As we add more quarters to the training set, do the estimated weight parameters remain stable?

In [ ]:
theta1_path = np.array(theta_path['theta1'])
theta2_path = np.array(theta_path['theta2'])
beta_path   = np.array(theta_path['beta'])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

time_ax = np.arange(len(theta1_path))

ax = axes[0]
ax.plot(time_ax, theta1_path, color='steelblue', linewidth=1.5)
ax.axhline(np.mean(theta1_path), color='tomato', linestyle='--', linewidth=1.5,
           label=f'Mean = {np.mean(theta1_path):.3f}')
ax.fill_between(time_ax,
                theta1_path - np.std(theta1_path),
                theta1_path + np.std(theta1_path),
                alpha=0.15, color='steelblue')
ax.set_xlabel('Training sample size (quarters)')
ax.set_ylabel('θ₁ estimate')
ax.set_title('θ₁ Over Expanding Window')
ax.legend(fontsize=9)

ax2 = axes[1]
ax2.plot(time_ax, theta2_path, color='tomato', linewidth=1.5)
ax2.axhline(np.mean(theta2_path), color='steelblue', linestyle='--', linewidth=1.5,
            label=f'Mean = {np.mean(theta2_path):.3f}')
ax2.fill_between(time_ax,
                 theta2_path - np.std(theta2_path),
                 theta2_path + np.std(theta2_path),
                 alpha=0.15, color='tomato')
ax2.set_xlabel('Training sample size (quarters)')
ax2.set_ylabel('θ₂ estimate')
ax2.set_title('θ₂ Over Expanding Window')
ax2.legend(fontsize=9)

ax3 = axes[2]
ax3.plot(time_ax, beta_path, color='#2ca02c', linewidth=1.5)
ax3.axhline(np.mean(beta_path), color='gray', linestyle='--', linewidth=1.5,
            label=f'Mean = {np.mean(beta_path):.3f}')
ax3.axhline(0, color='black', linewidth=0.8, alpha=0.5)
ax3.fill_between(time_ax,
                 beta_path - np.std(beta_path),
                 beta_path + np.std(beta_path),
                 alpha=0.15, color='#2ca02c')
ax3.set_xlabel('Training sample size (quarters)')
ax3.set_ylabel('β estimate')
ax3.set_title('β (IP coefficient) Over Expanding Window')
ax3.legend(fontsize=9)

plt.suptitle('MIDAS Parameter Stability Over Expanding Window', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('parameter_stability.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"\nParameter stability summary:")
print(f"  theta1: mean={np.mean(theta1_path):.3f}, std={np.std(theta1_path):.3f}")
print(f"  theta2: mean={np.mean(theta2_path):.3f}, std={np.std(theta2_path):.3f}")
print(f"  beta:   mean={np.mean(beta_path):.4f}, std={np.std(beta_path):.4f}")

## 5. RMSE Decomposition by Quarter

Which quarters contribute most to the forecast error? Are errors concentrated in crisis periods?

In [ ]:
# Compute squared error contribution by observation
sq_err_0 = np.array(err_0)**2
sq_err_2 = np.array(err_2)**2

# Sort by error magnitude
top_n = 10
top_idx = np.argsort(sq_err_0)[::-1][:top_n]

print(f"Top {top_n} largest 3-month nowcast errors:")
print(f"{'Rank':>5} {'Obs idx':>8} {'Actual':>10} {'Nowcast':>10} {'Error':>10} {'SqErr':>10}")
print("-" * 60)
for rank, idx in enumerate(top_idx, 1):
    actual = forecast_panel[0]['y_act'][idx]
    y_hat_ = forecast_panel[0]['y_hat'][idx]
    err_   = actual - y_hat_
    print(f"{rank:>5} {idx+MIN_TRAIN:>8} {actual:>10.4f} {y_hat_:>10.4f} {err_:>10.4f} {err_**2:>10.4f}")

# Contribution of top N errors to total RMSE
total_sq_err = np.nansum(sq_err_0)
top_sq_err   = np.sum(sq_err_0[top_idx])
print(f"\nTop {top_n} errors account for {top_sq_err/total_sq_err*100:.1f}% of total squared error")

In [ ]:
# --- RMSE excluding largest-error quarters ---
def rmse_excluding_top(errors, n_exclude):
    """RMSE excluding the n_exclude largest-error observations."""
    valid = ~np.isnan(errors)
    errs = errors[valid]
    sorted_idx = np.argsort(errs**2)[::-1]
    keep_idx = sorted_idx[n_exclude:]
    return np.sqrt(np.mean(errs[keep_idx]**2))


err_arr_0 = np.array(err_0)
print("RMSE Sensitivity to Outlier Exclusion (3-month vintage):")
print(f"{'Excl. obs':>12} {'RMSE':>10} {'Change vs baseline':>20}")
print("-" * 45)
base_rmse = rmse_excluding_top(err_arr_0, 0)
for n_excl in [0, 1, 2, 3, 5]:
    rmse_ex = rmse_excluding_top(err_arr_0, n_excl)
    change  = (rmse_ex - base_rmse) / base_rmse * 100
    n_remain = np.sum(~np.isnan(err_arr_0)) - n_excl
    print(f"{n_excl:>12} {rmse_ex:>10.4f} {change:>+20.1f}%  (n={n_remain})")

print("\nIf RMSE drops substantially when excluding 1-2 obs, results are driven by outliers (e.g., COVID).")

In [ ]:
# --- Final summary visualization ---
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Panel 1: Absolute errors by quarter
ax = axes[0, 0]
ax.bar(time_axis, np.abs(err_arr_0), color='steelblue', alpha=0.7, width=0.8,
       label='|Error| 3-month')
# Highlight top-5 errors
top5 = np.argsort(err_arr_0**2)[::-1][:5]
ax.bar(top5, np.abs(err_arr_0[top5]), color='tomato', alpha=0.9, width=0.8,
       label='Top-5 errors')
ax.axhline(base_rmse, color='black', linestyle='--', linewidth=1.5,
           label=f'RMSE={base_rmse:.4f}')
ax.set_xlabel('Out-of-sample observation')
ax.set_ylabel('|Forecast error| (%)')
ax.set_title('Absolute Forecast Error by Quarter')
ax.legend(fontsize=9)

# Panel 2: theta2 path
ax2 = axes[0, 1]
ax2.plot(time_ax, theta2_path, color='tomato', linewidth=1.5)
ax2.axhline(np.mean(theta2_path), color='gray', linestyle='--', linewidth=1.5)
ax2.set_xlabel('Expanding window')
ax2.set_ylabel('θ₂ estimate')
ax2.set_title('θ₂ Stability (back-loading parameter)')

# Panel 3: 3-month vs 1-month vintage comparison
ax3 = axes[1, 0]
valid = ~(np.isnan(err_arr_0) | np.isnan(err_2))
ax3.scatter(err_arr_0[valid], np.array(err_2)[valid], alpha=0.6, s=30, color='steelblue')
lim = max(np.abs(err_arr_0[valid]).max(), np.abs(np.array(err_2)[valid]).max()) * 1.1
ax3.plot([-lim, lim], [-lim, lim], 'k--', linewidth=1, alpha=0.5, label='y=x (equal errors)')
ax3.axhline(0, color='gray', linewidth=0.5)
ax3.axvline(0, color='gray', linewidth=0.5)
ax3.set_xlabel('3-month error (%)')
ax3.set_ylabel('1-month error (%)')
ax3.set_title('3-month vs 1-month Forecast Errors')
ax3.legend(fontsize=9)

# Panel 4: RMSE by vintage bar
ax4 = axes[1, 1]
vintage_names = ['1-month', '2-month', '3-month']
rmse_by_vintage = []
for h_idx, h in enumerate([2, 1, 0]):
    errs = np.array(forecast_panel[h]['error'])
    valid = ~np.isnan(errs)
    rmse_by_vintage.append(np.sqrt(np.mean(errs[valid]**2)))

bars = ax4.bar(vintage_names, rmse_by_vintage, color=['#2ca02c', '#ff7f0e', '#1f77b4'], alpha=0.85)
for bar, rmse in zip(bars, rmse_by_vintage):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{rmse:.4f}', ha='center', fontsize=10, fontweight='bold')
ax4.set_ylabel('RMSE')
ax4.set_title('Nowcast RMSE by Vintage Point')
ax4.set_ylim(0, max(rmse_by_vintage) * 1.25)

plt.suptitle('GDP Nowcast Evaluation Summary', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('nowcast_evaluation.png', dpi=120, bbox_inches='tight')
plt.show()
print("Summary figure saved.")

## 6. Self-Check

In [ ]:
print("Self-check...")
passed = 0
total = 5

# 1. Forecast errors have zero mean (unbiased in expectation)
err0_valid = err_arr_0[~np.isnan(err_arr_0)]
mean_err = np.mean(err0_valid)
assert abs(mean_err) < 0.5, \
    f"FAIL 1: Mean forecast error ({mean_err:.4f}) should be close to 0 (unbiased)"
passed += 1
print(f"  PASS 1: Mean forecast error = {mean_err:.4f} (approximately unbiased)")

# 2. 3-month RMSE <= 2-month RMSE (more months = better)
rmse_0 = rmse_by_vintage[2]  # 3-month
rmse_2 = rmse_by_vintage[0]  # 1-month
assert rmse_0 <= rmse_2 * 1.05, \
    f"FAIL 2: 3-month RMSE ({rmse_0:.4f}) should be <= 1-month RMSE ({rmse_2:.4f})"
passed += 1
print(f"  PASS 2: 3-month RMSE ({rmse_0:.4f}) <= 1-month RMSE ({rmse_2:.4f})")

# 3. Theta path has sensible range (0.5 < theta < 20)
assert all(0.1 < t < 20 for t in theta2_path), \
    f"FAIL 3: Some theta2 values outside sensible range [0.1, 20]"
passed += 1
print(f"  PASS 3: theta2 path stays in sensible range [{theta2_path.min():.2f}, {theta2_path.max():.2f}]")

# 4. Top-5 errors account for a substantial fraction of total squared error
top5_frac = np.sum(sq_err_0[top5]) / np.nansum(sq_err_0)
assert top5_frac > 0.15, \
    f"FAIL 4: Top-5 errors should account for >15% of total, got {top5_frac:.1%}"
passed += 1
print(f"  PASS 4: Top-5 errors account for {top5_frac:.1%} of total squared error")

# 5. RMSE excluding top-2 outliers is lower
rmse_excl2 = rmse_excluding_top(err_arr_0, 2)
assert rmse_excl2 <= base_rmse, \
    f"FAIL 5: RMSE excluding top-2 ({rmse_excl2:.4f}) should be <= baseline ({base_rmse:.4f})"
passed += 1
print(f"  PASS 5: Excluding top-2 reduces RMSE: {base_rmse:.4f} -> {rmse_excl2:.4f}")

print(f"\n{'='*40}")
print(f"Self-check: {passed}/{total} passed")
if passed == total:
    print("All checks passed.")

## Summary

| Analysis | Key Finding |
|----------|-------------|
| RMSE by vintage | 3-month < 2-month < 1-month (more data = better) |
| Error outliers | A few extreme quarters dominate total MSE |
| Parameter stability | theta path relatively stable; some variation during crises |
| Revision analysis | Month-3 release typically produces larger revisions when beta is front-loaded |

### Course Completion

You have now completed Module 03. You can:
- Build MIDAS nowcasts at three vintage points (ragged-edge handling)
- Run expanding-window evaluation and compute RMSE by vintage
- Construct forecast evolution charts
- Diagnose parameter stability and outlier sensitivity

**Module 04:** Dynamic Factor Models for nowcasting with many indicators simultaneously.